In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy as sp
from sklearn import linear_model
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
pa_gs = pd.read_csv('data/pa_data.csv')

coastal_pa_stations = pa_gs[pa_gs['dist_atlantic_km']< 150]
lakeside_pa_stations = pa_gs[pa_gs['dist_greatlakes_km']< 150]
inland_pa_stations = pa_gs[pa_gs['dist_coast_km']>150]

In [2]:
y = ['growing_season_length']
X = ['dtr_annual', 'dtr_spring', 'dtr_summer', 'tmax_annual', 'prcp_annual_mm', 'prcp_growing_season_mm', 
    'prcp_spring_mm', 'latitude', 'longitude', 'elevation_m', 'dist_coast_km', 'dist_greatlakes_km', 'dist_atlantic_km',
    'oni_annual', 'nao_annual', 'nao_djf', 'pna_annual', 'amo_annual', 'ohc700_atlantic', 'ohc700_atlantic_se', 
    'ohc700_north_atlantic', 'ohc700_north_atlantic_se', 'ohc700_south_atlantic', 'ohc700_south_atlantic_se', 
    'ohc2000_north_atlantic', 'ohc700_pacific', 'ohc700_world','ohc700_natl_djf', 'ohc700_natl_amj', 'sst_north_atlantic',
    'sst_gulf_stream', 'sst_gulf_mexico','sst_tropical_north_atl', 'pwat_eastern_us', 'pwat_southeast_us', 
    'pwat_northeast_us', 'pwat_gulf_coast', 'pwat_station', 'dewpoint_2m_eastern_us', 'soil_moisture_eastern_us',
    'cloud_cover_eastern_us', 'evaporation_eastern_us', 'dewpoint_2m_southeast_us', 'soil_moisture_southeast_us',
    'cloud_cover_southeast_us', 'evaporation_southeast_us', 'dewpoint_2m_northeast_us', 'soil_moisture_northeast_us',
    'cloud_cover_northeast_us', 'evaporation_northeast_us', 'dewpoint_2m_pennsylvania', 'soil_moisture_pennsylvania',
    'cloud_cover_pennsylvania', 'evaporation_pennsylvania', 'dewpoint_station', 'soil_moisture_station',
    'cloud_cover_station', 'evaporation_station']

In [44]:
def remove_nulls(data_subset, X, y):
        # Drop rows where any of the specified X columns have nulls
        data_subset_cleaned = data_subset.dropna(subset=X, ignore_index=True)
        # Separate the cleaned dataframe back into X and y
        X_cleaned = data_subset_cleaned[X]
        y_cleaned = data_subset_cleaned[y]
        return X_cleaned, y_cleaned

def multiple_regression(data_subset, input_variables, predicted_variable):
    #performs a standardized multiple linear regression to create a model of y based on the variables within X.
    # Required Libraries: r2score, mean_absolute_error, and mean_squared_error from sklearn.metrics, 
    # train_test_split from sklearn.model_selection, linear_model from sklearn, and StandardScaler from sklearn.preprocessing

    X_cleaned, y_cleaned = remove_nulls(data_subset, input_variables, predicted_variable)

    #counts total data points included in regression
    tot_data_points = len(X_cleaned)
    
    #splits data into training and testing groups
    X_train, X_test, y_train, y_test = train_test_split(X_cleaned, y_cleaned, test_size = 0.2, random_state =0)

    #converts y_train, y_test, and y_cleaned into series
    y_train= y_train.squeeze()
    y_test= y_test.squeeze()
    y_cleaned = y_cleaned.squeeze()
    
    #scales input data to standardize to mean of 0 and standard deviation of 1
    sc = StandardScaler()
    X_train_scaled = sc.fit_transform(X_train)
    X_test_scaled = sc.fit_transform(X_test)
    
    #creates multiple regression model based on training data
    regr = linear_model.LinearRegression()
    regr.fit(X_train_scaled, y_train)

    #calculates mean absolute error, mean squared error, and r squared score based on the predicted y
    y_predicted = regr.predict(X_test_scaled)
    mae = mean_absolute_error(y_test, y_predicted)
    mse = mean_squared_error(y_test, y_predicted)
    rmse = np.sqrt(mse)
    r_2_score = r2_score(y_test, y_predicted)
    n = len(y_cleaned)
    p = X_cleaned.shape[1]
    adj_r2_score = 1 - ((1 - r_2_score) * (n - 1) / (n - p - 1))
    y_std = y_cleaned.std()
    coefficients = pd.DataFrame(zip(X_cleaned.columns, X_cleaned.std(), regr.coef_))

    coefficients.columns = ['variable', 'var standard dev', 'coefficient']
    
    results = [f'MAE = {mae}', f'MSE = {mse}', f'RMSE = {rmse}', f'R Squared = {r_2_score}', 
               f'Adj. R Squared = {adj_r2_score}', f'Total Data Points = {tot_data_points}',
               f'(Reference) predicted variable standard deviation = {y_std}', coefficients]
    
    return display(results)

In [45]:
multiple_regression(pa_gs, X, y)

['MAE = 8.653954401679119',
 'MSE = 118.54336216146942',
 'RMSE = 10.887762036409017',
 'R Squared = 0.802483321255483',
 'Adj. R Squared = 0.7956683495914842',
 'Total Data Points = 1740',
 '(Reference) predicted variable standard deviation = 25.128478100584672',
                       variable  var standard dev  coefficient
 0                   dtr_annual          1.162618    -9.448399
 1                   dtr_spring          1.463406    -0.783348
 2                   dtr_summer          1.351448     1.657061
 3                  tmax_annual          1.650279     8.417943
 4               prcp_annual_mm        224.681732   -16.306781
 5       prcp_growing_season_mm        184.123407    20.479302
 6               prcp_spring_mm         85.879985     3.166995
 7                     latitude          0.672409   -11.853968
 8                    longitude          1.749712     7.593680
 9                  elevation_m        158.326516    -1.458976
 10               dist_coast_km         60

In [48]:
#removed all X variables with coefficient < |1|
X_influential = ['dtr_annual', 'dtr_summer', 'tmax_annual', 'prcp_annual_mm', 'prcp_growing_season_mm', 
    'prcp_spring_mm', 'latitude', 'longitude', 'elevation_m', 'dist_coast_km', 'dist_greatlakes_km', 'dist_atlantic_km',
    'oni_annual', 'nao_annual', 'nao_djf', 'pna_annual', 'amo_annual', 'ohc700_atlantic', 'ohc700_atlantic_se', 
    'ohc700_south_atlantic', 'ohc2000_north_atlantic', 'ohc700_pacific', 'ohc700_world','ohc700_natl_djf', 'ohc700_natl_amj', 
    'sst_north_atlantic','sst_gulf_stream', 'pwat_eastern_us','pwat_northeast_us', 'pwat_gulf_coast', 'pwat_station',
    'cloud_cover_eastern_us',  'dewpoint_2m_southeast_us','evaporation_southeast_us', 'dewpoint_2m_northeast_us', 
    'soil_moisture_northeast_us','cloud_cover_northeast_us', 'dewpoint_2m_pennsylvania','evaporation_pennsylvania', 
    'dewpoint_station', 'cloud_cover_station']
multiple_regression(pa_gs, X_influential, y)

['MAE = 8.674296355470146',
 'MSE = 118.47757695625499',
 'RMSE = 10.884740555302868',
 'R Squared = 0.8025929324138605',
 'Adj. R Squared = 0.7978263306641363',
 'Total Data Points = 1740',
 '(Reference) predicted variable standard deviation = 25.128478100584672',
                       variable  var standard dev  coefficient
 0                   dtr_annual          1.162618   -10.279544
 1                   dtr_summer          1.351448     1.818887
 2                  tmax_annual          1.650279     8.367813
 3               prcp_annual_mm        224.681732   -16.398181
 4       prcp_growing_season_mm        184.123407    20.472531
 5               prcp_spring_mm         85.879985     3.244283
 6                     latitude          0.672409   -11.718103
 7                    longitude          1.749712     7.284249
 8                  elevation_m        158.326516    -1.464093
 9                dist_coast_km         60.521467    -1.634493
 10          dist_greatlakes_km         9

In [43]:
# again, removed all X variables with coefficient < |1| from first refined model
X_influential_2 = ['dtr_annual', 'dtr_summer', 'tmax_annual', 'prcp_annual_mm', 'prcp_growing_season_mm', 
    'prcp_spring_mm', 'latitude', 'longitude', 'elevation_m', 'dist_coast_km', 'dist_greatlakes_km', 'dist_atlantic_km',
    'oni_annual', 'nao_annual', 'nao_djf', 'pna_annual', 'amo_annual', 'ohc700_atlantic', 
    'ohc700_south_atlantic', 'ohc2000_north_atlantic', 'ohc700_pacific', 'ohc700_world','ohc700_natl_djf', 'ohc700_natl_amj', 
    'sst_north_atlantic','sst_gulf_stream', 'pwat_eastern_us','pwat_northeast_us', 'pwat_gulf_coast', 'pwat_station',
    'dewpoint_2m_southeast_us','evaporation_southeast_us', 'dewpoint_2m_northeast_us', 
    'dewpoint_2m_pennsylvania','evaporation_pennsylvania', 
    'dewpoint_station', 'cloud_cover_station']
multiple_regression(pa_gs, X_influential_2, y)

['MAE = 8.67312897059619',
 'MSE = 118.4691159883503',
 'RMSE = 10.884351886462984',
 'R Squared = 0.8026070300592203',
 'Adj. R Squared = 0.7983158785387685',
 '(Reference) predicted variable standard deviation = 25.128478100584672',
                     variable  var standard dev  coefficient
 0                 dtr_annual          1.162618   -10.279544
 1                 dtr_summer          1.351448     1.818887
 2                tmax_annual          1.650279     8.367813
 3             prcp_annual_mm        224.681732   -16.398181
 4     prcp_growing_season_mm        184.123407    20.472531
 5             prcp_spring_mm         85.879985     3.244283
 6                   latitude          0.672409   -11.718103
 7                  longitude          1.749712     7.284249
 8                elevation_m        158.326516    -1.464093
 9              dist_coast_km         60.521467    -1.634493
 10        dist_greatlakes_km         91.537050   -10.389573
 11          dist_atlantic_km    

In [49]:
#removing all x variables with coefficient < |1.5| from 2nd refined model
X_influential_3 = ['dtr_annual', 'dtr_summer', 'tmax_annual', 'prcp_annual_mm', 'prcp_growing_season_mm', 
    'prcp_spring_mm', 'latitude', 'longitude', 'dist_coast_km', 'dist_greatlakes_km', 'dist_atlantic_km',
    'oni_annual', 'nao_annual', 'nao_djf', 'pna_annual', 'amo_annual', 'ohc700_atlantic', 
    'ohc700_south_atlantic', 'ohc2000_north_atlantic', 'ohc700_pacific', 'ohc700_world','ohc700_natl_djf', 'ohc700_natl_amj', 
    'sst_north_atlantic','sst_gulf_stream', 'pwat_eastern_us', 'pwat_gulf_coast', 'pwat_station',
    'dewpoint_2m_northeast_us', 'dewpoint_2m_pennsylvania','evaporation_pennsylvania', 'dewpoint_station', 'cloud_cover_station']
multiple_regression(pa_gs, X_influential_3, y)

['MAE = 8.705701705583804',
 'MSE = 119.92426986972434',
 'RMSE = 10.950994012861313',
 'R Squared = 0.8001824559922244',
 'Adj. R Squared = 0.7963172866180998',
 'Total Data Points = 1740',
 '(Reference) predicted variable standard deviation = 25.128478100584672',
                     variable  var standard dev  coefficient
 0                 dtr_annual          1.162618   -10.098319
 1                 dtr_summer          1.351448     1.625791
 2                tmax_annual          1.650279     9.578128
 3             prcp_annual_mm        224.681732   -16.565525
 4     prcp_growing_season_mm        184.123407    20.563189
 5             prcp_spring_mm         85.879985     3.312616
 6                   latitude          0.672409   -10.149974
 7                  longitude          1.749712     7.232032
 8              dist_coast_km         60.521467    -1.490184
 9         dist_greatlakes_km         91.537050    -9.545822
 10          dist_atlantic_km        133.471354     3.480082
 1

In [50]:
#again, removing all x variables with coefficient < |1.5| from 3nd refined model
X_influential_4 = ['dtr_annual', 'dtr_summer', 'tmax_annual', 'prcp_annual_mm', 'prcp_growing_season_mm', 
    'prcp_spring_mm', 'latitude', 'longitude', 'dist_greatlakes_km', 'dist_atlantic_km',
    'oni_annual', 'nao_djf', 'pna_annual', 'amo_annual', 'ohc700_atlantic', 
    'ohc2000_north_atlantic', 'ohc700_pacific', 'ohc700_world', 'ohc700_natl_amj', 
    'sst_north_atlantic','sst_gulf_stream', 'pwat_eastern_us', 'pwat_gulf_coast', 'pwat_station',
    'dewpoint_2m_northeast_us', 'dewpoint_2m_pennsylvania','evaporation_pennsylvania', 'cloud_cover_station']
multiple_regression(pa_gs, X_influential_4, y)

['MAE = 8.70360653978436',
 'MSE = 120.27008643481092',
 'RMSE = 10.966771924080984',
 'R Squared = 0.7996062572228855',
 'Adj. R Squared = 0.7963268739395662',
 'Total Data Points = 1740',
 '(Reference) predicted variable standard deviation = 25.128478100584672',
                     variable  var standard dev  coefficient
 0                 dtr_annual          1.162618    -9.830344
 1                 dtr_summer          1.351448     1.397806
 2                tmax_annual          1.650279     9.635027
 3             prcp_annual_mm        224.681732   -16.422984
 4     prcp_growing_season_mm        184.123407    20.348842
 5             prcp_spring_mm         85.879985     3.380801
 6                   latitude          0.672409   -11.649260
 7                  longitude          1.749712    11.966555
 8         dist_greatlakes_km         91.537050   -13.751093
 9           dist_atlantic_km        133.471354     5.966689
 10                oni_annual          0.597267    -2.206270
 11

In [51]:
X_choice_removals = ['dtr_annual', 'dtr_spring', 'dtr_summer', 'tmax_annual', 'prcp_annual_mm', 'prcp_growing_season_mm', 
    'prcp_spring_mm', 'latitude', 'longitude', 'elevation_m', 'dist_coast_km', 'dist_greatlakes_km', 'dist_atlantic_km',
    'ohc700_atlantic', 'ohc700_atlantic_se', 'ohc700_north_atlantic', 'ohc700_north_atlantic_se', 
    'ohc2000_north_atlantic', 'ohc700_pacific', 'ohc700_world','ohc700_natl_djf', 'ohc700_natl_amj', 'sst_north_atlantic',
    'sst_gulf_stream', 'sst_tropical_north_atl', 'pwat_eastern_us', 'pwat_northeast_us', 'pwat_station', 
    'dewpoint_2m_eastern_us', 'soil_moisture_eastern_us','cloud_cover_eastern_us', 'evaporation_eastern_us', 
    'dewpoint_2m_northeast_us', 'soil_moisture_northeast_us','cloud_cover_northeast_us', 'evaporation_northeast_us', 
    'dewpoint_2m_pennsylvania', 'soil_moisture_pennsylvania','cloud_cover_pennsylvania', 'evaporation_pennsylvania', 
    'dewpoint_station', 'soil_moisture_station','cloud_cover_station', 'evaporation_station']
multiple_regression(pa_gs, X_choice_removals, y)

['MAE = 8.592882842339472',
 'MSE = 117.67640440745765',
 'RMSE = 10.847875571164044',
 'R Squared = 0.8121023214237777',
 'Adj. R Squared = 0.8074471987563487',
 'Total Data Points = 1821',
 '(Reference) predicted variable standard deviation = 25.141863728241326',
                       variable  var standard dev  coefficient
 0                   dtr_annual          1.155768    -9.333436
 1                   dtr_spring          1.460778    -0.528394
 2                   dtr_summer          1.346237     1.448907
 3                  tmax_annual          1.659399     8.129434
 4               prcp_annual_mm        223.527298   -16.002349
 5       prcp_growing_season_mm        182.839232    19.991546
 6               prcp_spring_mm         86.815135     3.519209
 7                     latitude          0.672226   -13.176511
 8                    longitude          1.745983     9.947726
 9                  elevation_m        158.332507    -2.257715
 10               dist_coast_km         6

In [52]:
X_choice_removals_2 = ['dtr_annual', 'dtr_spring', 'dtr_summer', 'tmax_annual', 'prcp_annual_mm', 'prcp_growing_season_mm', 
    'prcp_spring_mm','ohc700_atlantic', 'ohc700_north_atlantic','ohc2000_north_atlantic', 'ohc700_pacific', 'ohc700_world',
    'ohc700_natl_djf', 'ohc700_natl_amj', 'sst_north_atlantic','sst_gulf_stream', 'sst_tropical_north_atl', 
    'pwat_eastern_us', 'pwat_northeast_us', 'pwat_station', 'dewpoint_2m_eastern_us', 'soil_moisture_eastern_us',
    'cloud_cover_eastern_us', 'evaporation_eastern_us', 'dewpoint_2m_northeast_us', 'soil_moisture_northeast_us',
    'cloud_cover_northeast_us', 'evaporation_northeast_us', 'dewpoint_2m_pennsylvania', 'soil_moisture_pennsylvania',
    'cloud_cover_pennsylvania', 'evaporation_pennsylvania', 'dewpoint_station', 'soil_moisture_station',
    'cloud_cover_station', 'evaporation_station']
multiple_regression(pa_gs, X_choice_removals_2, y)

['MAE = 8.644245374293895',
 'MSE = 120.37631279908649',
 'RMSE = 10.97161395598143',
 'R Squared = 0.8077912913433636',
 'Adj. R Squared = 0.8039126402718171',
 'Total Data Points = 1821',
 '(Reference) predicted variable standard deviation = 25.141863728241326',
                       variable  var standard dev  coefficient
 0                   dtr_annual          1.155768    -8.606068
 1                   dtr_spring          1.460778    -0.506074
 2                   dtr_summer          1.346237     0.726503
 3                  tmax_annual          1.659399    10.046342
 4               prcp_annual_mm        223.527298   -15.981489
 5       prcp_growing_season_mm        182.839232    19.539481
 6               prcp_spring_mm         86.815135     3.818024
 7              ohc700_atlantic          1.624986    -5.572113
 8        ohc700_north_atlantic          0.805715    -0.888291
 9       ohc2000_north_atlantic          1.281594    -0.043785
 10              ohc700_pacific          1

In [53]:
X_choice_removals_3 = ['dtr_annual', 'dtr_spring', 'tmax_annual', 'prcp_annual_mm', 'prcp_growing_season_mm', 
    'ohc700_atlantic', 'ohc2000_north_atlantic', 'ohc700_pacific', 'ohc700_world',
    'ohc700_natl_djf', 'ohc700_natl_amj', 'sst_north_atlantic','sst_gulf_stream', 
    'pwat_eastern_us', 'pwat_northeast_us', 'pwat_station', 'dewpoint_2m_northeast_us', 'soil_moisture_northeast_us',
    'cloud_cover_northeast_us', 'evaporation_northeast_us', 'dewpoint_2m_pennsylvania', 'soil_moisture_pennsylvania',
    'cloud_cover_pennsylvania', 'evaporation_pennsylvania', 'dewpoint_station', 'soil_moisture_station',
    'cloud_cover_station', 'evaporation_station']
multiple_regression(pa_gs, X_choice_removals_3, y)

['MAE = 8.643157423435815',
 'MSE = 124.18634195539767',
 'RMSE = 11.143892585420843',
 'R Squared = 0.8028814526775045',
 'Adj. R Squared = 0.7998031931543423',
 'Total Data Points = 1822',
 '(Reference) predicted variable standard deviation = 25.134966409407763',
                       variable  var standard dev  coefficient
 0                   dtr_annual          1.155695    -7.196557
 1                   dtr_spring          1.460669    -1.935689
 2                  tmax_annual          1.659636    10.898093
 3               prcp_annual_mm        225.050988   -12.456746
 4       prcp_growing_season_mm        183.348893    18.089798
 5              ohc700_atlantic          1.624955    18.785144
 6       ohc2000_north_atlantic          1.281618     9.528567
 7               ohc700_pacific          1.663640     2.789051
 8                 ohc700_world          3.946870   -25.171254
 9              ohc700_natl_djf          0.841334   -37.787609
 10             ohc700_natl_amj          

In [54]:
multiple_regression(inland_pa_stations, X, y)

['MAE = 7.903038615421362',
 'MSE = 101.94476197632694',
 'RMSE = 10.096769878348567',
 'R Squared = 0.7925760441220471',
 'Adj. R Squared = 0.782068979108142',
 'Total Data Points = 1204',
 '(Reference) predicted variable standard deviation = 22.683614070734397',
                       variable  var standard dev  coefficient
 0                   dtr_annual          1.096584    -8.635601
 1                   dtr_spring          1.429845    -0.893934
 2                   dtr_summer          1.273667     1.651865
 3                  tmax_annual          1.493089     4.707797
 4               prcp_annual_mm        218.895351   -17.111908
 5       prcp_growing_season_mm        180.815703    21.427992
 6               prcp_spring_mm         83.766335     3.011035
 7                     latitude          0.585141    -9.348587
 8                    longitude          1.481903     6.981162
 9                  elevation_m        148.665453    -2.810526
 10               dist_coast_km         40

In [56]:
#removing stations with coefficient < |1|
X_inland_influential = ['dtr_annual', 'dtr_summer', 'tmax_annual', 'prcp_annual_mm', 'prcp_growing_season_mm', 
    'prcp_spring_mm', 'latitude', 'longitude', 'elevation_m', 'dist_greatlakes_km', 'dist_atlantic_km',
    'oni_annual', 'nao_annual', 'nao_djf', 'pna_annual', 'ohc700_atlantic', 'ohc700_south_atlantic', 
    'ohc2000_north_atlantic', 'ohc700_pacific', 'ohc700_world','ohc700_natl_djf', 'ohc700_natl_amj', 'sst_north_atlantic',
    'sst_gulf_stream', 'pwat_eastern_us', 'pwat_northeast_us', 'pwat_gulf_coast', 'pwat_station',
    'cloud_cover_eastern_us', 'dewpoint_2m_southeast_us', 'soil_moisture_southeast_us',
    'cloud_cover_southeast_us', 'evaporation_southeast_us', 'dewpoint_2m_northeast_us', 'soil_moisture_northeast_us',
    'dewpoint_2m_pennsylvania', 'evaporation_pennsylvania', 'dewpoint_station',
    'cloud_cover_station']
multiple_regression(inland_pa_stations, X_inland_influential, y)

['MAE = 8.60777795535519',
 'MSE = 114.71368871659519',
 'RMSE = 10.710447643147097',
 'R Squared = 0.7353517715353731',
 'Adj. R Squared = 0.7269055514779914',
 'Total Data Points = 1262',
 '(Reference) predicted variable standard deviation = 22.686285420837336',
                       variable  var standard dev  coefficient
 0                   dtr_annual          1.089919    -8.785166
 1                   dtr_summer          1.269761     0.845724
 2                  tmax_annual          1.510218     5.954168
 3               prcp_annual_mm        216.788465   -17.844578
 4       prcp_growing_season_mm        179.037872    21.105820
 5               prcp_spring_mm         85.247407     3.932680
 6                     latitude          0.584685    -4.744891
 7                    longitude          1.479806     6.925138
 8                  elevation_m        148.652185    -1.666182
 9           dist_greatlakes_km         67.252531    -0.856105
 10            dist_atlantic_km        101

In [57]:
#removing stations with coefficient < |1| again
X_inland_influential_2 = ['dtr_annual', 'dtr_summer', 'tmax_annual', 'prcp_annual_mm', 'prcp_growing_season_mm', 
    'prcp_spring_mm', 'latitude', 'longitude', 'elevation_m', 'dist_greatlakes_km', 'dist_atlantic_km',
    'oni_annual', 'nao_djf', 'pna_annual', 'ohc700_atlantic', 'ohc700_south_atlantic', 
    'ohc2000_north_atlantic', 'ohc700_pacific', 'ohc700_world','ohc700_natl_djf', 'ohc700_natl_amj', 'sst_north_atlantic',
    'sst_gulf_stream', 'pwat_eastern_us', 'pwat_northeast_us', 'pwat_gulf_coast', 'pwat_station',
    'cloud_cover_eastern_us', 'dewpoint_2m_southeast_us', 'soil_moisture_southeast_us','dewpoint_2m_northeast_us',
    'dewpoint_2m_pennsylvania', 'evaporation_pennsylvania', 'dewpoint_station', 'cloud_cover_station']
multiple_regression(inland_pa_stations, X_inland_influential_2, y)

['MAE = 8.62517235180058',
 'MSE = 114.99612659176056',
 'RMSE = 10.723624694652482',
 'R Squared = 0.7347001781279072',
 'Adj. R Squared = 0.7271263659211182',
 'Total Data Points = 1262',
 '(Reference) predicted variable standard deviation = 22.686285420837336',
                       variable  var standard dev  coefficient
 0                   dtr_annual          1.089919    -8.785166
 1                   dtr_summer          1.269761     0.845724
 2                  tmax_annual          1.510218     5.954168
 3               prcp_annual_mm        216.788465   -17.844578
 4       prcp_growing_season_mm        179.037872    21.105820
 5               prcp_spring_mm         85.247407     3.932680
 6                     latitude          0.584685    -4.744891
 7                    longitude          1.479806     6.925138
 8                  elevation_m        148.652185    -1.666182
 9           dist_greatlakes_km         67.252531    -0.856105
 10            dist_atlantic_km        101

In [58]:
X_inland_choice_removals = ['dtr_annual', 'dtr_spring', 'dtr_summer', 'tmax_annual', 'prcp_annual_mm', 'prcp_growing_season_mm', 
    'prcp_spring_mm', 'latitude', 'longitude', 'elevation_m', 'dist_coast_km', 'dist_greatlakes_km', 'dist_atlantic_km',
    'ohc700_atlantic', 'ohc700_atlantic_se', 'ohc700_north_atlantic', 'ohc700_north_atlantic_se', 
    'ohc2000_north_atlantic', 'ohc700_pacific', 'ohc700_world','ohc700_natl_djf', 'ohc700_natl_amj', 'sst_north_atlantic',
    'sst_gulf_stream', 'sst_tropical_north_atl', 'pwat_eastern_us', 'pwat_northeast_us', 'pwat_station', 
    'dewpoint_2m_eastern_us', 'soil_moisture_eastern_us','cloud_cover_eastern_us', 'evaporation_eastern_us', 
    'dewpoint_2m_northeast_us', 'soil_moisture_northeast_us','cloud_cover_northeast_us', 'evaporation_northeast_us', 
    'dewpoint_2m_pennsylvania', 'soil_moisture_pennsylvania','cloud_cover_pennsylvania', 'evaporation_pennsylvania', 
    'dewpoint_station', 'soil_moisture_station','cloud_cover_station', 'evaporation_station']
multiple_regression(inland_pa_stations, X_inland_choice_removals, y)

['MAE = 8.648164992194724',
 'MSE = 116.01220677753429',
 'RMSE = 10.770896284782166',
 'R Squared = 0.7323560479360225',
 'Adj. R Squared = 0.722679520499034',
 'Total Data Points = 1262',
 '(Reference) predicted variable standard deviation = 22.686285420837336',
                       variable  var standard dev  coefficient
 0                   dtr_annual          1.089919    -6.916648
 1                   dtr_spring          1.428828    -1.650671
 2                   dtr_summer          1.269761     0.366134
 3                  tmax_annual          1.510218     5.867576
 4               prcp_annual_mm        216.788465   -17.661168
 5       prcp_growing_season_mm        179.037872    21.216615
 6               prcp_spring_mm         85.247407     3.896407
 7                     latitude          0.584685    -4.884597
 8                    longitude          1.479806     4.930749
 9                  elevation_m        148.652185    -1.731376
 10               dist_coast_km         40